# Test Technique - Data Scientist Full Stack

1. Le notebook suivant exploitent les données contenues dans le sous-dossier `./data`. Veillez à placer les fichiers csv source dans ce sous-dossier
- jobs.csv
- users.csv
- users_history.csv
- feedbacks.csv
- test_users.csv
2. Les scripts sont en python3. La seule librairie utilisée est:
- pandas

## Objectif

L'objectif est d’entraîner un modèle capable de prédire un taux de compatibilité entre un utilisateur et une offre d'emploi, en fonction de son parcours et des caractéristiques de l'offre (features).

Pour cela, la data set feedbacks est fondamental. On y trouve des **couples users/jobs** (features X) qui nous donnent un **résultat** *"viewed, applied, hired"* (y) que l'on cherchera à reproduire. On pourra donc entraîner notre modèle sur ces couples.

En phase de prédiction, nous fournirons au modèle une table de tous les couples possibles **test_users/jobs** pour obtenir un score et sélectionner les offres d'emploi qui obtiendront les meilleurs scores pour chaque user, ou aucun si le score passe en dessous d'un certain seuil. 

Nous évaluerons d'abord la qualité et les caractéristiques des données pour identifier un modèle qui est compatible avec nos objectifs. Nous identifierons ensuite les données utiles à notre modèle, ce qui guidera également notre nettoyage et transformation des données. Enfin, nous évoquerons les stratégies d'entrainement, d'évaluation et les risques éventuels. Cet exercie a pour but d'imaginer un modèle fonctionnel en peu de temps. Nous utiliserons donc un minimum de données pour cela pour n'avoir que peu de preprocessing.

## Qualité des données

Pour évaluer la qualité des données, les caractérstiques du dataset suivantes sont évaluées

1. **L'unicité des clés primaires** pour `jobs`, `users` et `test_users`(tables dimensions)
2. **Le pourcentage de nullité**: plus la donnée est riche, plus le modèle sera performant. On pourrait decider d'un seuil au dessus duquel nous n'utilisons pas cette colonne car trop vide, ou décider d'une stratégie de remplissage pertinente.
3. **Cardinalité d'une variable**, soit le nombre des valeurs distinctes. Si elle est haute dans une variable discrèete repérée à risque, cela peut etre un signe qu'il y a beaucoup d'erreurs de saisie. Si il y en a trop, cela montrera la nécessité de la réduire, qui nécessitera beaucoup de preprocessing.

Le script suivant met en oeuvre cette stratégie pour chaque table:

In [1]:
import pandas as pd

In [2]:
### Fonction donnant un overview de la quality des datasets
def quality_dim_table(df,key):  
    working_df=df.copy()
    quality_df=pd.DataFrame(columns=['df_col','%nulls','uniques','type'])
    quality_df['df_col']=working_df.columns
    
    ### 1. Non duplicité des clés primaires
    if key:
        duplicates=working_df[key].duplicated().sum()
        if duplicates:
            print("### Duplicated IDs ###\nduplicates\n")
        else:
            print('### No duplicated IDs###\n')

    ### 2. Nullité
    nulls_list=[]
    for col in working_df:
        nulls_list.append(working_df[col].isnull().sum()/len(working_df)*100)
    quality_df['%nulls']=nulls_list
    
    ### Cardinalité et risques de frappes + types
    uniques_list=[]
    types_list=[]
    for col in working_df:
        uniques_list.append(working_df[col].nunique())
        types_list.append(working_df[col].dtype)
    quality_df['uniques']=uniques_list
    quality_df['type']=types_list
    
    print("### Quality assessment ###\n", quality_df)
            
    return 

In [3]:
quality_dim_table(pd.read_csv('./data/users.csv'),key='UserID')

### No duplicated IDs###

### Quality assessment ###
                   df_col     %nulls  uniques     type
0                 UserID   0.000000     1337    int64
1                   City   0.000000      294      str
2                  State   0.000000        1      str
3                Country   0.000000        1      str
4                ZipCode   0.074794      392      str
5             DegreeType  25.280479        6      str
6                  Major  23.111444      485      str
7         GraduationDate  29.169783      243      str
8       WorkHistoryCount   0.000000       13    int64
9   TotalYearsExperience   3.066567       45  float64
10     CurrentlyEmployed  13.836948        2      str
11         ManagedOthers   0.000000        2      str
12        ManagedHowMany   0.000000       57    int64
13       MajorCategoryID  23.036649       80  float64


In [4]:
quality_dim_table(pd.read_csv('./data/test_users.csv'),key='UserID')

### No duplicated IDs###

### Quality assessment ###
                   df_col     %nulls  uniques     type
0                 UserID   0.000000      277    int64
1                   City   0.000000       97      str
2                  State   0.000000        1      str
3                Country   0.000000        1      str
4                ZipCode   0.000000      146    int64
5             DegreeType  31.768953        6      str
6                  Major  31.407942      120      str
7         GraduationDate  31.407942       96      str
8       WorkHistoryCount   0.000000       12    int64
9   TotalYearsExperience   4.693141       34  float64
10     CurrentlyEmployed  16.606498        2      str
11       MajorCategoryID  57.400722       29  float64


In [5]:
quality_dim_table(pd.read_csv('./data/jobs.csv'),key='JobID')

### No duplicated IDs###

### Quality assessment ###
            df_col     %nulls  uniques     type
0           JobID   0.000000     4291    int64
1           Title   0.000000     3432      str
2     Description   0.000000     4112      str
3    Requirements   6.199021     3308      str
4            City   0.000000      585      str
5           State   0.000000       37      str
6         Country   0.000000        1      str
7            Zip5  27.848986      769  float64
8       StartDate   0.000000     4291      str
9         EndDate   0.000000       42      str
10  JobCategoryID   0.000000      195    int64
11    apply_count   0.000000       51  float64
12    hired_count   0.000000        9  float64


In [6]:
quality_dim_table(pd.read_csv('./data/users_history.csv'),key=False)

### Quality assessment ###
           df_col  %nulls  uniques     type
0         UserID     0.0     1549    int64
1       Sequence     0.0       13    int64
2       JobTitle     0.0     4685      str
3  JobCategoryID     0.0      197  float64


In [7]:
quality_dim_table(pd.read_csv('./data/feedbacks.csv'),key=False)

### Quality assessment ###
    df_col  %nulls  uniques   type
0  UserID     0.0     1861  int64
1   JobID     0.0     3149  int64
2   Event     0.0        3    str
3    Date     0.0      425    str


- Aucune clé primaire en double pour les tables de dimension
- Certaines colonnes contiennent beaucoup de champs vides. Il faudra donc soit décider si nous souhaitons les conserver, ou trouver une stratégie pour les remplir sans biaiser notre modèle.
- On voit ici que la colonne à risques `Major` de `users` contient 486 majors distincts, visiblement renseignés à la main. On pourrait y appliquer une fonction de detection de similitudes pour estimer le nombre de majors très similaires qui doivent être fusionnés. Cela représente beaucoup de preprocessing alors que la colonne `MajorCategoryID` contient aussi des information sur les majors.
- Il faut bien s'assurer que la cardinalité de certaines valeurs clées sont bien ce à quoi on s'attend (*eg Card(Event) = 3*), sinon il faut etablir un stratégie d'eviction des erreurs.

## Caractéristiques des données et modèle

- **quantité de données:** Ici nous avons un data set de quelques milliers de lignes (<100k)
- **qualité des données:** il y un nombre non négligeable de champs nos renseignés
- **données structurés:** tabulaires classiques dans les milieux RH
- **Types de données et distribution:** nous avons une grande majorité de variables discrètes (comme les catégories). Les quelques données quantitatives ne sont pas distribuées normalement (*eg. beaucoup de user avec 1 an d'expérience, peu avec 15*).

Nous n'avons pas un grande quantité de données, avec beaucoup de variable qualitatives et quelques variables quantitatives non normales. Cela nous mène à penser que des modèles comme les arbres de décisions sont parfaitement indiqués, que l'on peut utiliser en classification ou en régression, sur des données structurées non normales. Le **XGBoost** est un modèle robuste qui sait gérer les données manquantes. Avec une stratégie mieux établie des remplacement des données manquantes, le RandomForest est un autre modèle plus simple qui conviendrait.
<br />Modèle pour la suite: **XGBoost**, nous pouvons donc conserver des series avec des données manquantes si aucune stratégie évidente se dégage.

## Quelles données conserver et comment les transformer?

L'idée sera de créer une matrice d’interaction entre toutes les features pertinentes des users et des offres d'emploi. Nous souhaitons ici architecturer un modèle de recommandation simple en peu de temps, donc certaines features ne seront pas incluses car trop lourdes à preprocess pour l'instant.

### Users et test_users:

| A conserver ou transformer                                   | A éliminer ou considérer en V2                               |
| ------------------------------------------------------------ | ------------------------------------------------------------ |
| - **UserID**: les IDs ne doivent pas entraîner le modèle mais sont necessaire pour joindre les tables<br />- **DegreeType**: basse cardinalité -> à encoder en catégorie numérique <br />- **MajorCategoryID** : faciles à utiliser telle quelle pour un arbre, haute cardinalité mais à tester en V1 <br />- **GraduationDate**: la transformer en nombre de jours écoulé depuis le diplôme pour l’entraînement (**GraduationDays**) <br />- **WorkHistoryCount**, **TotalYearsExperience**, **CurrentlyEmployed**, **ManagedOthers**, **ManagedHowMany** sont des variables numériques déjà formatés correctement ou presque. | -  **Major:** Il y plus de 400 valeurs différentes, dont beaucoup d'erreurs de saisie et de majors très proches les unes des autres. Il faudrait les regrouper en catégories plus large et identifier les erreurs. Pour un premier modèle, MajorCategoryID donne une information déja pertinente.<br />- Données géographiques: il y a peu ou pas de variations dans **Country** et **State**. **ZipCode** pourrait être comparé à ceux des offres d'emploi dans un modèle amélioré |

### Jobs

| A conserver ou transformer                                   | A éliminer ou considérer en V2                               |
| ------------------------------------------------------------ | ------------------------------------------------------------ |
| - **JobID** : conserver pour jointure<br />- **JobCategoryID**: variable discrete idéal pour un arbre | - **City**, **State**, **Country, Zip5** : idem que pour jobs (pourraient être utilisés si plus de variations dans les futures données)<br />- **Description** et **Requirements** : il y a beaucoup d'informations dans ces textes qui seraient très intéressant d'extraire et utiliser, en cherchant des mots-clés ou en utilisant dd'autres algorithmes. Pour une V1, cela dépasse le scope de ce projet. <br />- **StartDate**, **EndDate** : ici il s'agit des dates d'apparition de l'offre. La start date pourra etre utilisée (en jours) pour traduire une certaine fraicheur de l'offre face à laquelle les users ne sont pas indiférents. <br />- **Title**: pourrait être remplacé par le maximum des scores de similitude avec les noms des postes dans l'historique des users afin d'identifier si le user a un experience similaire |

### Users_history

| A conserver ou transformer                                   | A éliminer ou considérer en V2                               |
| ------------------------------------------------------------ | ------------------------------------------------------------ |
| - **UserID**: conserver pour jointure<br />- **JobCategoryID**: il existe environ 200 ID différent. Une premiere approche est de comparer la liste des categories dans lesquelles le useres a travaillé avec la categorie du job en question pour créer une variable booléenne **HasWorkedInTargetCategoryID** qui sera facile à utiliser par un arbre<br /> | - **Sequence**: l'importance d'un poste récemment occupé pourrait etre valorisée dans un modèle plus évolué<br />- **JobTitle** : chaque JobTitle pourrait être comparé avec le nom du poste de l'offre pour résumé en un score maximum de similitude |

### Feedbacks

| A conserver ou transformer:                                  | A éliminer ou considérer en V2 |
| ------------------------------------------------------------ | ------------------------------ |
| - **UserID** : conserver pour jointure<br />- **JobID** : conserver pour jointure<br />- **Event** : notre target class à conserver absolument | - **Date**                     |



## Exemple de resultat 
![etl](etl.jpeg)
Une fois :
- les données transformées et jointes,
- les colonnes unutiles supprimées, 
- les valeurs discrètes encodées,
<br /><br /> l'objectif est d'obtenir une table d'interaction de la forme suivante:

| DegreeType | MajorCategoryID | GraduationDays | WorkHistoryCount | TotalYearsExperience | CurrentlyEmployed | ManagedOthers | ManagedHowMany | HasWorkedInTargetJobCategory | Event |
| ---------- | --------------- | -------------- | ---------------- | -------------------- | ----------------- | ------------- | -------------- | ---------------------------- | ----- |
| 3          | 24              | 567            | 2                | 5                    | 0                 | 1             | 12             | 1                            | 0.25  |


## Nettoyage
Nous connaissons désormais nos données utiles donc nous effectuer leur nettoyage

### Users et test_users
Le script suivant nettoie les colonnes d'intérêt de notre set puis supprime celles non utilisées par la suite
1. Degree type: normaliser et remplir les cases vides
2. GraduationDate: s'assurer que toutes les dates sont au meme format, remplir les dates vides par la médiane pour créer un colonne **GraduationDays** qui sera la différence avec DATE_REF, que notre modèle saura gérer.
3. WorkHistoryCount: remplacer le vide par 0
4. TotalYearsExperience: remplacer le vide par 0
5. CurrentlyEmployed: normaliser les données et transformer en booléen. Aucune stratégie de remplacement
6. ManagedOthers: normaliser les données et transformer en booléen. Aucune stratégie de remplacement
7. ManagedHowMany: nombre entier
8. MajorCategoryID: nombre entier

In [8]:
#Date de référence pour calculer un nombre de jours depuis X
DATE_REF=pd.to_datetime('2026-01-01')

def transform_users(df_users):
    clean_df=df_users.copy()
        
    ### 1. DEGREE TYPE
    clean_df['DegreeType']=clean_df['DegreeType']\
        .fillna('')\
        .astype(str)\
        .str.lower()\
        .str.strip()\
        .replace(['nan','na',''],'not applicable')

    ### 2. GRADUATION DATE
    # Normaliser dates
    clean_df['GraduationDate']=pd.to_datetime(clean_df['GraduationDate'],\
                                        dayfirst=True,
                                        errors='coerce')
    # Remplir dates vides par médiane pour calculer GraduationDays
    mediane_date = clean_df['GraduationDate'].median()      
    clean_df['GraduationDate'] = clean_df['GraduationDate'].fillna(mediane_date)
    #Créer une collone en jours depuis début de l'année pour modèle
    clean_df['GraduationDays']=-1*(clean_df['GraduationDate']-DATE_REF).dt.days
  
  
    ### 3. WORK HISTORY COUNT
    clean_df['WorkHistoryCount']=clean_df['WorkHistoryCount']\
            .astype('Int64')\
            .fillna(0)
    
    ### 4. YEARS EXPERIENCE
    clean_df['TotalYearsExperience']=clean_df['TotalYearsExperience']\
            .astype('Int64')\
            .fillna(0)
    
    ### 5. CURRENTLY EMPLOYED
    clean_df['CurrentlyEmployed']=clean_df['CurrentlyEmployed']\
            .str.strip()\
            .str.lower()\
            .map({'yes':True,'no':False,
                  'y':True,'n':False})\
            .astype('boolean')
    
    ### 6. MANAGED OTHERS
    #'if' because missing cols in test_users.csv
    if 'ManagedOthers' in clean_df:
        clean_df['ManagedOthers']=clean_df['ManagedOthers']\
                .str.strip()\
                .str.lower()\
                .map({'yes':True,'no':False,
                    'y':True,'n':False})\
                .astype('boolean')
    
    ### 7. MANAGEDHOWMANY
    if 'ManagedHowMany' in clean_df:
        clean_df['ManagedHowMany']=clean_df['ManagedHowMany']\
            .astype('Int64')      
    
    ### 8. MAJOR CATEGORY
    clean_df['MajorCategoryID']=clean_df['MajorCategoryID']\
        .astype('Int64')
    
    #Removing columns not used in the model
    clean_df=clean_df.drop(['City','State','Country','ZipCode','Major','GraduationDate',], axis=1)
    
    clean_df.info()
    return clean_df


Cette fonction exporte en csv et affiche un résumé des information de la table:

In [9]:
def load_csv(df, filename):
    df.to_csv(filename, index=False)
    print(f"\n### {filename} created ###")

In [10]:
clean_users=transform_users(pd.read_csv('./data/users.csv'))
load_csv(clean_users,'clean_users.csv')

<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   UserID                1337 non-null   int64  
 1   DegreeType            1337 non-null   str    
 2   WorkHistoryCount      1337 non-null   Int64  
 3   TotalYearsExperience  1337 non-null   Int64  
 4   CurrentlyEmployed     1152 non-null   boolean
 5   ManagedOthers         1337 non-null   boolean
 6   ManagedHowMany        1337 non-null   Int64  
 7   MajorCategoryID       1029 non-null   Int64  
 8   GraduationDays        1337 non-null   int64  
dtypes: Int64(4), boolean(2), int64(2), str(1)
memory usage: 83.7 KB

### clean_users.csv created ###


Idem avec `test_users.csv` (qui ne contient pas les colonnes ManagedOthers et ManagedHowMany)

In [11]:
clean_test_users=transform_users(pd.read_csv('./data/test_users.csv'))
load_csv(clean_users,'clean_test_users.csv')

<class 'pandas.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   UserID                277 non-null    int64  
 1   DegreeType            277 non-null    str    
 2   WorkHistoryCount      277 non-null    Int64  
 3   TotalYearsExperience  277 non-null    Int64  
 4   CurrentlyEmployed     231 non-null    boolean
 5   MajorCategoryID       118 non-null    Int64  
 6   GraduationDays        277 non-null    int64  
dtypes: Int64(3), boolean(1), int64(2), str(1)
memory usage: 14.5 KB

### clean_test_users.csv created ###


### Jobs
Deux colonnes à conserver et bien au format entier
1. JobID
2. JobCategoryID

In [12]:
def transform_jobs(df):
    #Extrait les colonnes d'intérêt
    clean_df=df[['JobID', 'JobCategoryID']]
    #Making sure INT
    clean_df['JobCategoryID']=clean_df['JobCategoryID'].astype('Int64')
    
    clean_df.info()
    return clean_df

In [13]:
clean_jobs=transform_jobs(pd.read_csv('./data/jobs.csv'))
load_csv(clean_jobs,'clean_jobs.csv')

<class 'pandas.DataFrame'>
RangeIndex: 4291 entries, 0 to 4290
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   JobID          4291 non-null   int64
 1   JobCategoryID  4291 non-null   Int64
dtypes: Int64(1), int64(1)
memory usage: 71.4 KB

### clean_jobs.csv created ###


### Users history
S'assurer du format des categorie et grouper les users pour qu'il n'occupent qu'une ligne avec leur experiences passées.


In [21]:
def transform_history(df):
    working_df=df.copy()
    #INT
    working_df['JobCategoryID']=working_df['JobCategoryID'].astype(int) 
    
    #Group JobCategoryID by users and transform to set
    history_series = working_df.groupby('UserID')['JobCategoryID'].apply(list) #/!\userID devient index
    #Turn into DataFrame again
    clean_df = history_series.reset_index(name='PastJobCategories')
    
    clean_df.info()
    return clean_df

In [22]:
clean_history=transform_history(pd.read_csv('./data/users_history.csv'))
load_csv(clean_history,'clean_history.csv')

<class 'pandas.DataFrame'>
RangeIndex: 1549 entries, 0 to 1548
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   UserID             1549 non-null   int64 
 1   PastJobCategories  1549 non-null   object
dtypes: int64(1), object(1)
memory usage: 24.3+ KB

### clean_history.csv created ###


### Feedbacks
Tout conserver sauf la colonne `Date` et normaliser les `Event`.

In [16]:
def transform_feedbacks(df):
    clean_df= df.copy()
    clean_df=clean_df.drop('Date',axis=1)
    clean_df['Event']=clean_df['Event']\
        .astype(str)\
        .str.lower()\
        .str.strip()
        
    clean_df.info()
    return clean_df

In [17]:
clean_feedbacks=transform_feedbacks(pd.read_csv('./data/feedbacks.csv'))
load_csv(clean_feedbacks,'clean_feedbacks.csv')

<class 'pandas.DataFrame'>
RangeIndex: 28928 entries, 0 to 28927
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   UserID  28928 non-null  int64
 1   JobID   28928 non-null  int64
 2   Event   28928 non-null  str  
dtypes: int64(2), str(1)
memory usage: 678.1 KB

### clean_feedbacks.csv created ###


## Entrainement


Pour créer la training table, il nous reste à
1. Joindre les tables `clean_jobs`, `users`, `clean_history` et `feedbacks`
2. Alimenter la nouvelle colonne `HasWorkedInTargetJobCategory`
3. Supprimer les colonnes inutiles
4. Attribué des poids différents à chaque événement (un *hired* a plus de poids qu'un *viewed*)

Nous avons un set près à être decoupé en sets d'entrainement et de test, entrainé, fine tuné et appliqué.
Pour etablir nos prédiction, nous devons former une "to_predict table" en croisant tous les candidats avec toutes les offres ouvertes et y appliquer notre modèle prédictif comme ceci:

![training](training.jpeg)

Attention: il faut aussi alimenter la colonne "HasWorkedInTaregt" pour le set d'entraienement!!!

## Stratégies d'évaluation

Pour optimiser le modèle on pourra chercher comment la valeur des poids influe positivement ou negativement sur les metriques.

Pour un système de recommendation XGBoost, ces premières metrqiues donnent un bon apercu de la qualité du modèle:
- **AUC-ROC**: Un score AUC-ROC élevé témoigne de la capacité globale du modèle à ordonner correctement les profils, c'est-à-dire à placer les interactions fortes (comme *Hired* ou *Applied*) au-dessus des interactions faibles ou des absences d'intérêt (*Viewed*).
- **Matrice de confusion:** Une fois le modèle entraîné, ses prédictions brutes se présentent sous la forme d'un score de probabilité continu compris entre 0 et 1. La matrice de confusion 4x4 intervient pour calibrer nos règles métiers post-entraînement.

## Risques et limites
1. **Le produit cartésien** de tous les users et toutes les offres peut vite créer une table gigantesque. En fonction du nombre d'individus à tester, il faut considérer un pré-filtrage des données.
2. **Biais de sélection**: la table feedbacks ne recense que les évenement où un utilisateur a au moins vu l'offre. Nous ne connaissons pas toutes les fois où un user à ignoré une offre. Notre modèle ici va attribuer des étiquette artificiellement élevée puisqu'il vit dans un monde ou tout le monde d'interesse à tout. On pourrait inserer des faux négatifs pour contrebalancer.
3. Beaucoup d'information utiles ont été supprimées des tables par soucis de simplicité. Un modèle plus évolué devra les prendre encompte comme mentionné dans le tableau de nettoyage.

## Clonclusion
Les tables sont désormais épurées et prêtes à être fusionnées pour entrainer notre XGBoost, dans une première version fonctionnelle mais pas optimale car beaucoup d'informations contenues dans le dataset ont été mises de côté. Lors de ce genre d'exercice, le preprocessing est la phase cruciale pour obtenir des prédictions qui reflètent réellement la donnée.